# ETF/债券 3秒快照数据检查与时间范围汇总

本 Notebook 统一完成以下任务：
1. 读取多个目录下全部 CSV 文件；
2. 用一个 cell 输出每个目录下所有文件名；
3. 检查每个 CSV 的数据列是否一致（缺失列 / 冗余列）；
4. 统计每个 CSV 的 `trade_time` 最小值和最大值；
5. 产出适合后续数据合并与清洗的数据结构。

In [ ]:
from pathlib import Path
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 220)

In [ ]:
# ===== 1) 基础配置：目录与预期列 =====
# 说明：同时保留上一版 ETF 目录，并新增本次要求的 3 个债券目录
base_dirs = {
    # 上一版 ETF 目录
    '511090': Path('国债ETF数据/3秒快照/511090'),
    '511130': Path('国债ETF数据/3秒快照/511130'),

    # 本次新增债券目录
    '019776': Path('债券数据/3秒快照/019776'),
    '019742': Path('债券数据/3秒快照/019742'),
    '019789': Path('债券数据/3秒快照/019789'),
}

expected_columns = [
    'Unnamed: 0', 'code', 'trade_time', 'pre_close', 'last', 'open', 'high', 'low', 'close',
    'volume', 'amount', 'num_trades', 'high_limited', 'low_limited',
    'ask_price1', 'ask_volume1', 'bid_price1', 'bid_volume1',
    'ask_price2', 'ask_volume2', 'bid_price2', 'bid_volume2',
    'ask_price3', 'ask_volume3', 'bid_price3', 'bid_volume3',
    'ask_price4', 'ask_volume4', 'bid_price4', 'bid_volume4',
    'ask_price5', 'ask_volume5', 'bid_price5', 'bid_volume5',
    'iopv', 'trading_phase_code'
]

# 收集各目录下 CSV（排序保证输出稳定）
csv_files_by_dir = {k: sorted(v.glob('*.csv')) for k, v in base_dirs.items()}

# 检查目录是否存在，便于快速定位路径问题
dir_status_df = pd.DataFrame([
    {
        'folder_key': k,
        'path': str(v),
        'exists': v.exists(),
        'csv_count': len(csv_files_by_dir[k])
    }
    for k, v in base_dirs.items()
])
dir_status_df

In [ ]:
# ===== 2) 一个 cell 输出每个文件夹所有 CSV 文件名 =====
for folder_key, files in csv_files_by_dir.items():
    print(f'\n目录 {folder_key} - 文件数量: {len(files)}')
    for f in files:
        print('  -', f.name)

In [ ]:
# ===== 3) 检查每个 CSV 的列一致性 + trade_time 范围 =====
# 数据结构说明：
# report_df: 文件级别检查报告（后续可直接作为数据目录）
# dataframes_by_dir: 原始数据缓存，便于后续逐文件清洗与合并

records = []
dataframes_by_dir = {k: {} for k in base_dirs.keys()}

for folder_key, files in csv_files_by_dir.items():
    for file_path in files:
        df = pd.read_csv(file_path)
        dataframes_by_dir[folder_key][file_path.name] = df

        actual_cols = df.columns.tolist()
        missing_cols = [c for c in expected_columns if c not in actual_cols]
        extra_cols = [c for c in actual_cols if c not in expected_columns]

        # 统一解析 trade_time，异常值转 NaT，便于后续清洗
        if 'trade_time' in df.columns:
            tt = pd.to_datetime(df['trade_time'], errors='coerce')
            tt_min = tt.min()
            tt_max = tt.max()
            tt_nat_count = int(tt.isna().sum())
        else:
            tt_min = None
            tt_max = None
            tt_nat_count = None

        records.append({
            'folder': folder_key,
            'file_name': file_path.name,
            'row_count': len(df),
            'column_count': len(actual_cols),
            'is_column_match': (len(missing_cols) == 0 and len(extra_cols) == 0),
            'missing_columns': missing_cols,
            'extra_columns': extra_cols,
            'trade_time_min': tt_min,
            'trade_time_max': tt_max,
            'trade_time_nat_count': tt_nat_count,
        })

report_df = pd.DataFrame(records).sort_values(['folder', 'file_name']).reset_index(drop=True)
report_df

In [ ]:
# ===== 4) 分目录展示：列一致性检查结果 =====
for folder_key in base_dirs.keys():
    print(f'\n===== 目录 {folder_key} 列检查结果 =====')
    sub = report_df.loc[
        report_df['folder'] == folder_key,
        ['file_name', 'is_column_match', 'missing_columns', 'extra_columns']
    ]
    display(sub)

In [ ]:
# ===== 5) 分目录展示：每个文件 trade_time 最小值与最大值 =====
for folder_key in base_dirs.keys():
    print(f'\n===== 目录 {folder_key} trade_time 范围 =====')
    sub = report_df.loc[
        report_df['folder'] == folder_key,
        ['file_name', 'trade_time_min', 'trade_time_max', 'trade_time_nat_count', 'row_count']
    ]
    display(sub)

In [ ]:
# ===== 6) 汇总结构检查（便于后续 merge/清洗） =====
print('report_df shape:', report_df.shape)
print('dataframes_by_dir keys:', list(dataframes_by_dir.keys()))

for folder_key in dataframes_by_dir:
    print(f'{folder_key} 文件数:', len(dataframes_by_dir[folder_key]))